# Plantilla de evaluacion de prompts sobre el conjunto de validacion

Este cuaderno constituye la plantilla utilizada para evaluar los cinco prompts candidatos sobre el conjunto de **validacion**, con el fin de seleccionar, para cada modelo, el prompt de mejor desempeno (segun macro-F1) antes de la evaluacion oficial sobre el conjunto de test.

Para evaluar un modelo distinto, basta con actualizar los parametros de la seccion 1 (`MODEL_ID`, `MODEL_SHORT_NAME`, `MODEL_TYPE` y `SUPPORTS_ENABLE_THINKING`); el resto del cuaderno no requiere modificaciones.


## 1. Configuracion del modelo y parametros generales


In [ ]:
# CONFIGURACIÓN DEL MODELO
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct" # Este es el id del moedlo encontrado en Hugging Face (se obtiene de la url)
MODEL_SHORT_NAME = "llama3_1_8b_instruct" # Este nombre se usara para los archivos generados en este cuaderno

MODEL_TYPE = "causal_lm"              # cambiar dependiendo la arquitectura del modelo
SUPPORTS_ENABLE_THINKING = False      # indicar si el modelo posee thinking (como por ejemplo Qwen 3.5)

# Configuración general
SEED = 42
MAX_NEW_TOKENS = 10        # la respuesta esperada es solo la etiqueta, no necesitamos generar mucho texto
TEMPERATURE = 0.0          # determinismo: queremos la respuesta más probable, no variabilidad creativa
DO_SAMPLE = False          # con temperature=0 y do_sample=False, la generación es reproducible

RUTA_VAL = "/content/val_final.csv"
RUTA_TEST = "/content/test_final.csv"

CHECKPOINT_CADA = 200      # cada cuántos ejemplos se guarda un checkpoint parcial durante el loop


## 2. Instalacion de dependencias


In [ ]:
#!pip install -q "transformers==5.10.4" accelerate "kernels==0.10.3" # usar esto en el caso de Qwen 3.5. Para el resto de modelos es suficiente las siguientes lineas

import transformers
print("Versión de transformers instalada:", transformers.__version__)

import torch
print("Versión de torch instalada:", torch.__version__)
print("Versión de CUDA (torch):", torch.version.cuda)

#Para el resto de modelos (no Qwen 3.5) usar las siguientes versiones
#Versión de kernels instalada: 0.10.3
#Versión de torch instalada: 2.13.0+cu130


## 3. Verificacion de GPU


In [ ]:
print("GPU disponible:", torch.cuda.is_available())
print("Nombre GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")


## 4. Autenticacion en Hugging Face


In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Login exitoso")


## 5. Carga del conjunto de validacion


In [ ]:
import pandas as pd

df_val = pd.read_csv(RUTA_VAL)

print("Validation cargado:", df_val.shape)
print(df_val["label_name"].value_counts())
df_val.head()


## 6. Definicion de los cinco prompts candidatos


In [ ]:
PROMPTS = {
    1: {
        "system": None,
        "user": "Clasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negativo, neutral, positivo. Comentario: {text}"
    },
    2: {
        "system": None,
        "user": "Para el texto proporcionado, analiza su sentimiento y clasifícalo únicamente en una de las siguientes clases: negativo, neutral, positivo. Texto: {text}"
    },
    3: {
        "system": None,
        "user": "Clasifica el sentimiento del texto en español peruano informal en solo una de las siguientes categorías: negativo, neutral, positivo. Texto: {text}"
    },
    4: {
        "system": "Eres un clasificador de sentimientos de comentarios en español peruano. Responde únicamente con una de las siguientes clases: negativo, neutral, positivo.",
        "user": "Clasifica el siguiente comentario. Comentario: {text}"
    },
    5: {
        "system": "Eres un experto en clasificar el sentimiento de textos que pueden contener jergas o expresiones coloquiales del español peruano. Responde únicamente con una de las siguientes clases: negativo, neutral, positivo. No brindes texto adicional.",
        "user": "Clasifica el siguiente texto. Texto: {text}"
    },
}

def construir_mensajes(prompt_id: int, texto: str):
    """Devuelve la lista de mensajes (formato chat) para un prompt_id y un texto dado."""
    config = PROMPTS[prompt_id]
    mensajes = []
    if config["system"] is not None:
        mensajes.append({"role": "system", "content": config["system"]})
    mensajes.append({"role": "user", "content": config["user"].format(text=texto)})
    return mensajes


texto_ejemplo = df_val.iloc[0]["text_clean"]
for pid in PROMPTS:
    print(f"--- Prompt {pid} ---")
    print(construir_mensajes(pid, texto_ejemplo))
    print()


## 7. Carga del modelo

Esta celda se adapta automaticamente segun `MODEL_TYPE` definido en la Seccion 1.


In [ ]:
import time
import torch

print(f"Cargando modelo: {MODEL_ID}  (tipo: {MODEL_TYPE})")
t0 = time.time()

if MODEL_TYPE == "multimodal":
    from transformers import AutoProcessor, AutoModelForMultimodalLM

    processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=hf_token,
    )
    tokenizer = processor.tokenizer

elif MODEL_TYPE == "causal_lm":
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=hf_token,
    )
    processor = None  # no aplica para modelos de solo texto

else:
    raise ValueError(f"MODEL_TYPE debe ser 'multimodal' o 'causal_lm', recibido: {MODEL_TYPE}")

tiempo_carga = time.time() - t0

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print(f"Modelo cargado en {tiempo_carga:.1f} segundos")
print(f"Memoria GPU reservada tras cargar: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Memoria GPU asignada tras cargar: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
import random
import numpy as np

def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fijar_semilla(SEED)


## 8. Funcion de generacion de respuestas (unificada para ambos tipos de modelo)

Esta funcion abstrae la diferencia entre `processor.apply_chat_template` (multimodal) y `tokenizer.apply_chat_template` (causal_lm estandar), y entre modelos que soportan `enable_thinking` y los que no.


In [ ]:
def generar_respuesta(mensajes):
    kwargs_template = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    if SUPPORTS_ENABLE_THINKING:
        kwargs_template["enable_thinking"] = False

    if MODEL_TYPE == "multimodal":
        inputs = processor.apply_chat_template(mensajes, **kwargs_template).to(model.device)
    else:
        inputs = tokenizer.apply_chat_template(mensajes, **kwargs_template).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        )

    respuesta = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return respuesta


# Prueba rápida con un solo ejemplo antes de correr todo el dataset
mensajes_prueba = construir_mensajes(1, texto_ejemplo)
respuesta_prueba = generar_respuesta(mensajes_prueba)
print("--- Respuesta generada (prueba) ---")
print(repr(respuesta_prueba))


## 9. Funcion de parseo de la respuesta generada


In [ ]:
import re
import unicodedata

def normalizar_texto(texto: str) -> str:
    texto = texto.lower().strip()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto

def parsear_respuesta(respuesta: str) -> str:
    texto = normalizar_texto(respuesta)

    tiene_negativo = bool(re.search(r"\bnegativ", texto))
    tiene_neutral = bool(re.search(r"\bneutral", texto))
    tiene_positivo = bool(re.search(r"\bpositiv", texto))

    encontrados = [
        etiqueta for etiqueta, encontrado in
        [("negative", tiene_negativo), ("neutral", tiene_neutral), ("positive", tiene_positivo)]
        if encontrado
    ]

    if len(encontrados) == 1:
        return encontrados[0]
    else:
        return "invalido"


casos_prueba = ["positivo\n", "Negativo.", "La respuesta es: NEUTRAL", "no estoy seguro", "positivo o neutral"]
for caso in casos_prueba:
    print(f"{caso!r:40} -> {parsear_respuesta(caso)}")


## 10. Evaluacion de los cinco prompts sobre el conjunto de validacion


In [ ]:
!pip install -q tqdm


In [ ]:
import time
import torch
import pandas as pd
from tqdm.auto import tqdm

def evaluar_modelo_en_prompt(df, prompt_id, checkpoint_cada=CHECKPOINT_CADA, nombre_archivo=None):
    fijar_semilla(SEED)

    resultados = []
    torch.cuda.reset_peak_memory_stats()
    t_inicio = time.time()

    for idx, fila in tqdm(df.iterrows(), total=len(df), desc=f"Prompt {prompt_id}"):
        texto = fila["text_clean"]
        mensajes = construir_mensajes(prompt_id, texto)
        respuesta_cruda = generar_respuesta(mensajes)
        prediccion = parsear_respuesta(respuesta_cruda)

        resultados.append({
            "idx_original": idx,
            "text_clean": texto,
            "label_name": fila["label_name"],
            "prompt_id": prompt_id,
            "respuesta_cruda": respuesta_cruda,
            "prediccion": prediccion,
        })

        if nombre_archivo and (len(resultados) % checkpoint_cada == 0):
            pd.DataFrame(resultados).to_csv(nombre_archivo, index=False)

    tiempo_total = time.time() - t_inicio
    memoria_pico_gb = torch.cuda.max_memory_allocated() / 1e9

    df_resultados = pd.DataFrame(resultados)
    if nombre_archivo:
        df_resultados.to_csv(nombre_archivo, index=False)

    metadata = {
        "modelo": MODEL_ID,
        "prompt_id": prompt_id,
        "seed": SEED,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
        "n_ejemplos": len(df),
        "tiempo_total_seg": round(tiempo_total, 1),
        "tiempo_promedio_por_ejemplo_seg": round(tiempo_total / len(df), 3),
        "memoria_pico_gb": round(memoria_pico_gb, 2),
    }

    return df_resultados, metadata


In [ ]:
resultados_por_prompt = {}
log_experimentos = []

for prompt_id in PROMPTS:
    nombre_archivo = f"val_{MODEL_SHORT_NAME}_prompt{prompt_id}.csv"
    df_res, metadata = evaluar_modelo_en_prompt(
        df_val, prompt_id, checkpoint_cada=CHECKPOINT_CADA, nombre_archivo=nombre_archivo
    )
    resultados_por_prompt[prompt_id] = df_res
    log_experimentos.append(metadata)
    print(f"Prompt {prompt_id} terminado -> {metadata}")

df_log = pd.DataFrame(log_experimentos)
df_log.to_csv(f"log_experimentos_{MODEL_SHORT_NAME}_validation.csv", index=False)
df_log


## 11. Metricas por prompt y seleccion del mejor prompt para este modelo


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import pandas as pd

CLASES = ["negative", "neutral", "positive"]

def calcular_metricas(df_resultados, prompt_id):
    y_true = df_resultados["label_name"]
    y_pred = df_resultados["prediccion"]

    n_invalidos = (y_pred == "invalido").sum()
    pct_invalidos = round(100 * n_invalidos / len(df_resultados), 2)

    accuracy = accuracy_score(y_true, y_pred)  # robusto a etiquetas "invalido" fuera de CLASES

    reporte = classification_report(
        y_true, y_pred, labels=CLASES, output_dict=True, zero_division=0
    )
    macro_f1 = f1_score(y_true, y_pred, labels=CLASES, average="macro", zero_division=0)
    matriz = confusion_matrix(y_true, y_pred, labels=CLASES)

    resumen = {
        "prompt_id": prompt_id,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "precision_negative": round(reporte["negative"]["precision"], 4),
        "recall_negative": round(reporte["negative"]["recall"], 4),
        "f1_negative": round(reporte["negative"]["f1-score"], 4),
        "precision_neutral": round(reporte["neutral"]["precision"], 4),
        "recall_neutral": round(reporte["neutral"]["recall"], 4),
        "f1_neutral": round(reporte["neutral"]["f1-score"], 4),
        "precision_positive": round(reporte["positive"]["precision"], 4),
        "recall_positive": round(reporte["positive"]["recall"], 4),
        "f1_positive": round(reporte["positive"]["f1-score"], 4),
        "pct_invalidos": pct_invalidos,
    }
    return resumen, matriz


resumenes = []
matrices = {}

for prompt_id in PROMPTS:
    df_res = resultados_por_prompt[prompt_id]
    resumen, matriz = calcular_metricas(df_res, prompt_id)
    resumenes.append(resumen)
    matrices[prompt_id] = matriz

df_comparacion_prompts = pd.DataFrame(resumenes)
df_comparacion_prompts.to_csv(f"comparacion_prompts_{MODEL_SHORT_NAME}_validation.csv", index=False)

mejor_prompt = df_comparacion_prompts.loc[df_comparacion_prompts["macro_f1"].idxmax(), "prompt_id"]
print(f"Mejor prompt para {MODEL_ID} (según macro-F1 en validation): Prompt {mejor_prompt}")
df_comparacion_prompts.sort_values("macro_f1", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, prompt_id in enumerate(PROMPTS):
    sns.heatmap(
        matrices[prompt_id], annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASES, yticklabels=CLASES, ax=axes[i], cbar=False
    )
    axes[i].set_title(f"Prompt {prompt_id}")
    axes[i].set_xlabel("Predicción")
    if i == 0:
        axes[i].set_ylabel("Real")

plt.suptitle(f"Matrices de confusión por prompt — {MODEL_ID} (validation)")
plt.tight_layout()
plt.savefig(f"matrices_confusion_{MODEL_SHORT_NAME}_validation.png", dpi=150)
plt.show()
